# DeepVQE Training Explorer

Interactive walkthrough of the DeepVQE AEC pipeline. Each section visualizes
one component of the model.

**Requirements**: `pip install -e '.[dev]'` for `librosa` and `ipykernel`.

Works with or without a trained checkpoint — uses `DummyAECDataset` by default.

In [ ]:
# --- Setup ---
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

from src.config import load_config
from src.model import DeepVQEAEC
from src.stft import stft, istft, make_window
from src.losses import DeepVQELoss
from src.viz import (
    register_hooks, remove_hooks,
    plot_spectrogram_comparison, plot_delay_with_gt,
    plot_ccm_mask, plot_encoder_activations, plot_activation_stats,
)
from data.dataset import DummyAECDataset
from train import delay_samples_to_frame

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

# Load config
cfg = load_config("../configs/default.yaml")
device = torch.device("cpu")  # CPU is fine for exploration

# Create model
model = DeepVQEAEC.from_config(cfg).to(device)
model.eval()

# Optional: load checkpoint
CHECKPOINT = None  # Set to e.g. "../checkpoints/best.pt" to load weights
if CHECKPOINT:
    from train import load_checkpoint
    load_checkpoint(CHECKPOINT, model)
    print(f"Loaded checkpoint: {CHECKPOINT}")
else:
    print("Using untrained (random) weights")

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

## 1. Data Synthesis

Generate a dummy example and visualize the waveforms.

In [ ]:
# Create a dummy sample
sr = cfg.audio.sample_rate
delay_samples = 768  # ~48ms at 16kHz, exactly 3 hops

ds = DummyAECDataset(
    length=1,
    target_len=int(cfg.training.clip_length_sec * sr),
    n_fft=cfg.audio.n_fft,
    hop_length=cfg.audio.hop_length,
    delay_range=(delay_samples, delay_samples),
    sr=sr,
)
sample = ds[0]
meta = sample["metadata"]

# Plot waveforms
fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
target_len = sample["mic_wav"].shape[-1]
t = np.arange(target_len) / sr

axes[0].plot(t, sample["mic_wav"].numpy(), linewidth=0.5)
axes[0].set_title(f"Microphone (delay={meta['delay_samples']} samp, "
                  f"SNR={meta['snr_db']:.0f}dB, SER={meta['ser_db']:.0f}dB)")
axes[0].set_ylabel("Amplitude")

axes[1].plot(t, sample["clean_wav"].numpy(), linewidth=0.5, color="green")
axes[1].set_title("Clean (near-end)")
axes[1].set_ylabel("Amplitude")

# Far-end from STFT (we don't store farend_wav directly)
ref_wav = istft(sample["ref_stft"].unsqueeze(0), cfg.audio.n_fft, cfg.audio.hop_length,
                length=target_len).squeeze(0)
axes[2].plot(t, ref_wav.numpy(), linewidth=0.5, color="orange")
axes[2].set_title("Far-end Reference")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Amplitude")

# Mark delay
delay_sec = delay_samples / sr
for ax in axes:
    ax.axvline(delay_sec, color="red", linestyle="--", alpha=0.5, label=f"delay={delay_sec*1000:.1f}ms")
axes[0].legend(fontsize=8)

fig.tight_layout()
plt.show()

## 2. STFT & Feature Extraction

Show raw STFT magnitude/phase and power-law compressed version.

In [ ]:
mic_stft = sample["mic_stft"].unsqueeze(0)  # (1, F, T, 2)
ref_stft = sample["ref_stft"].unsqueeze(0)
clean_stft = sample["clean_stft"].unsqueeze(0)

# Raw magnitude and phase
mic_mag = torch.sqrt(mic_stft[..., 0]**2 + mic_stft[..., 1]**2 + 1e-12)
mic_phase = torch.atan2(mic_stft[..., 1], mic_stft[..., 0])

# Power-law compressed
from src.blocks import FE
fe = FE(c=cfg.model.power_law_c)
mic_fe = fe(mic_stft)  # (1, 2, T, F)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Raw magnitude (dB)
mag_db = 20 * torch.log10(mic_mag[0] + 1e-12).numpy()
im0 = axes[0, 0].imshow(mag_db, aspect="auto", origin="lower", cmap="magma")
axes[0, 0].set_title("Raw Magnitude (dB)")
fig.colorbar(im0, ax=axes[0, 0], label="dB")

# Phase
im1 = axes[0, 1].imshow(mic_phase[0].numpy(), aspect="auto", origin="lower", cmap="hsv")
axes[0, 1].set_title("Phase")
fig.colorbar(im1, ax=axes[0, 1], label="radians")

# Power-law compressed real/imag
im2 = axes[1, 0].imshow(mic_fe[0, 0].numpy(), aspect="auto", origin="lower", cmap="RdBu_r")
axes[1, 0].set_title(f"FE real (c={cfg.model.power_law_c})")
fig.colorbar(im2, ax=axes[1, 0])

im3 = axes[1, 1].imshow(mic_fe[0, 1].numpy(), aspect="auto", origin="lower", cmap="RdBu_r")
axes[1, 1].set_title(f"FE imag (c={cfg.model.power_law_c})")
fig.colorbar(im3, ax=axes[1, 1])

for ax in axes.flat:
    ax.set_xlabel("Frame")
    ax.set_ylabel("Freq bin")

fig.suptitle(f"sqrt-Hann window, n_fft={cfg.audio.n_fft}, hop={cfg.audio.hop_length}", y=1.01)
fig.tight_layout()
plt.show()

# Show the sqrt-Hann window
window = make_window(cfg.audio.n_fft).numpy()
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.plot(window)
ax.set_title("sqrt-Hann Analysis Window")
ax.set_xlabel("Sample")
ax.set_ylabel("Amplitude")
fig.tight_layout()
plt.show()

## 3. Encoder Walkthrough

Register hooks, run forward pass, show channel-mean activation heatmap at each encoder stage.

In [ ]:
activation_store, hook_handles = register_hooks(model)

with torch.no_grad():
    enhanced, delay_dist = model(mic_stft, ref_stft, return_delay=True)

print("Captured activations:")
for name, act in activation_store.items():
    print(f"  {name:15s} {tuple(act.shape)}")

# Show encoder activation heatmaps
fig = plot_encoder_activations(activation_store)
fig.suptitle("Encoder Stage Activations (channel mean)", y=1.02)
plt.show()

# Activation statistics
fig = plot_activation_stats(activation_store)
fig.suptitle("Per-Layer Activation Statistics", y=1.02)
plt.show()

## 4. AlignBlock Deep Dive

Examine the cross-attention mechanism: Q/K projections, similarity scores, delay distribution.

In [ ]:
# Get mic_enc2 and far_enc2 from activation store
mic_e2 = activation_store["mic_enc2"].to(device)  # (1, 128, T, 65)
far_e2 = activation_store["far_enc2"].to(device)

# Run AlignBlock manually to capture internals
align = model.align
with torch.no_grad():
    Q = align.pconv_mic(mic_e2)
    K = align.pconv_ref(far_e2)

print(f"Q shape: {Q.shape}  (B, H={align.hidden_channels}, T, F)")
print(f"K shape: {K.shape}")

# Show Q and K channel-mean heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
im0 = axes[0].imshow(Q[0].mean(0).cpu().numpy().T, aspect="auto", origin="lower", cmap="RdBu_r")
axes[0].set_title("Q (mic projection, channel mean)")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(K[0].mean(0).cpu().numpy().T, aspect="auto", origin="lower", cmap="RdBu_r")
axes[1].set_title("K (ref projection, channel mean)")
fig.colorbar(im1, ax=axes[1])

for ax in axes:
    ax.set_xlabel("Frame")
    ax.set_ylabel("Freq bin")
fig.tight_layout()
plt.show()

# Delay distribution
fig = plot_delay_with_gt(delay_dist, delay_samples, cfg.audio.hop_length, cfg.model.dmax)
fig.suptitle("AlignBlock Delay Distribution", y=1.02)
plt.show()

# Temperature sensitivity
print(f"\nCurrent temperature: {align.temperature}")
print(f"GT delay frame: {delay_samples_to_frame(torch.tensor([delay_samples]), cfg.audio.hop_length, cfg.model.dmax).item()}")
print(f"Predicted peak frame (mean over T): {delay_dist[0].mean(0).argmax().item()}")

In [ ]:
# Temperature sweep
import math

B, C, T, F = far_e2.shape
with torch.no_grad():
    Ku = align.unfold_k(K)
    Ku = Ku.view(B, align.hidden_channels, align.dmax, T, F)
    Ku = Ku.permute(0, 1, 3, 2, 4).contiguous()
    V = torch.sum(Q.unsqueeze(-2) * Ku, dim=-1) / math.sqrt(F)
    V = align.conv(V)  # (B, 1, T, dmax)

temps = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, temp in zip(axes.flat, temps):
    A = torch.softmax(V / temp, dim=-1)
    im = ax.imshow(A[0, 0].cpu().numpy().T, aspect="auto", origin="lower", cmap="viridis")
    peak = A[0, 0].mean(0).argmax().item()
    ax.set_title(f"T={temp}, peak={peak}")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Delay")

fig.suptitle("Temperature Sensitivity", fontsize=14)
fig.tight_layout()
plt.show()

## 5. Bottleneck GRU

Visualize GRU hidden state dynamics with PCA.

In [ ]:
bn_act = activation_store["bottleneck"]  # (1, 128, T, 9)
print(f"Bottleneck output shape: {bn_act.shape}")

# Flatten to (T, C*F) for PCA
bn_flat = bn_act[0].permute(1, 0, 2).reshape(bn_act.shape[2], -1).float()  # (T, 1152)
bn_centered = bn_flat - bn_flat.mean(0)

# SVD-based PCA (top 2 components)
U, S, Vt = torch.linalg.svd(bn_centered, full_matrices=False)
pca_2d = (U[:, :2] * S[:2]).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA trajectory
T_bn = pca_2d.shape[0]
colors = plt.cm.viridis(np.linspace(0, 1, T_bn))
axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=colors, s=15)
axes[0].plot(pca_2d[:, 0], pca_2d[:, 1], alpha=0.3, color="gray", linewidth=0.5)
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].set_title("GRU Hidden State PCA (colored by time)")
sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(0, T_bn))
fig.colorbar(sm, ax=axes[0], label="Frame")

# Mean activation per unit (first 64 of 1152)
mean_per_unit = bn_flat.mean(0).numpy()[:64]
axes[1].bar(range(len(mean_per_unit)), mean_per_unit, color="steelblue", width=1.0)
axes[1].set_xlabel("Unit index (first 64 of 1152)")
axes[1].set_ylabel("Mean activation")
axes[1].set_title("GRU Output Mean Activation")

fig.tight_layout()
plt.show()

# Variance explained
var_explained = (S**2) / (S**2).sum()
print(f"Top-5 PCA variance explained: {var_explained[:5].numpy()}")
print(f"Cumulative top-10: {var_explained[:10].sum():.1%}")

## 6. Decoder Walkthrough

Show channel-mean activations at each decoder stage, demonstrating frequency upsampling.

In [ ]:
dec_keys = [k for k in activation_store if k.startswith("dec")]
dec_keys.sort(key=lambda x: int(x.replace("dec", "")), reverse=True)

n = len(dec_keys)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]

for ax, key in zip(axes, dec_keys):
    act = activation_store[key][0]  # (C, T, F)
    hm = act.float().mean(0).numpy()  # (T, F)
    im = ax.imshow(hm.T, aspect="auto", origin="lower", cmap="viridis")
    ax.set_title(f"{key} ({act.shape[0]}ch, F={act.shape[2]})")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Freq bin")
    fig.colorbar(im, ax=ax)

fig.suptitle("Decoder Stages — Frequency Upsampling", y=1.02)
fig.tight_layout()
plt.show()

## 7. CCM Mask Analysis

Decompose the 27-channel mask into H_real/H_imag, show mask magnitude and phase.

In [ ]:
# dec1 output is the 27ch mask before CCM
mask_27 = activation_store["dec1"][0]  # (27, T, F)
print(f"Mask shape: {mask_27.shape}")

# Cube-root basis decomposition
v_real = torch.tensor([1.0, -0.5, -0.5])
v_imag = torch.tensor([0.0, np.sqrt(3)/2, -np.sqrt(3)/2])

m = mask_27.float().view(3, 9, mask_27.shape[1], mask_27.shape[2])
H_real = (v_real[:, None, None, None] * m).sum(0)  # (9, T, F)
H_imag = (v_imag[:, None, None, None] * m).sum(0)

# Magnitude and phase of H (mean over 9 kernel elements)
H_mag = torch.sqrt(H_real**2 + H_imag**2 + 1e-12).mean(0)  # (T, F)
H_phase = torch.atan2(H_imag.mean(0), H_real.mean(0))  # (T, F)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Raw 27ch mask (show first 3 channels as RGB-ish)
for i, (ax, ch_name) in enumerate(zip(axes[0], ["Basis 0 (ch 0-8)", "Basis 1 (ch 9-17)"])):
    ch_mean = mask_27[i*9:(i+1)*9].float().mean(0).numpy()  # (T, F)
    im = ax.imshow(ch_mean.T, aspect="auto", origin="lower", cmap="RdBu_r")
    ax.set_title(ch_name)
    fig.colorbar(im, ax=ax)

# Mask magnitude
im2 = axes[1, 0].imshow(H_mag.numpy().T, aspect="auto", origin="lower", cmap="viridis")
axes[1, 0].set_title("Mean |H| (mask magnitude)")
fig.colorbar(im2, ax=axes[1, 0])

# Mask phase
im3 = axes[1, 1].imshow(H_phase.numpy().T, aspect="auto", origin="lower", cmap="hsv")
axes[1, 1].set_title("Mean phase(H)")
fig.colorbar(im3, ax=axes[1, 1], label="radians")

for ax in axes.flat:
    ax.set_xlabel("Frame")
    ax.set_ylabel("Freq bin")

fig.suptitle("CCM Mask Decomposition", y=1.01)
fig.tight_layout()
plt.show()

# Ideal ratio mask comparison
clean_mag = torch.sqrt(clean_stft[0, ..., 0]**2 + clean_stft[0, ..., 1]**2 + 1e-12)
mic_mag_s = torch.sqrt(mic_stft[0, ..., 0]**2 + mic_stft[0, ..., 1]**2 + 1e-12)
irm = (clean_mag / (mic_mag_s + 1e-12)).clamp(0, 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
im0 = axes[0].imshow(H_mag.numpy().T, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=2)
axes[0].set_title("Predicted |H|")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(irm.numpy(), aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=2)
axes[1].set_title("Ideal Ratio Mask |clean|/|mic|")
fig.colorbar(im1, ax=axes[1])

for ax in axes:
    ax.set_xlabel("Frame")
    ax.set_ylabel("Freq bin")
fig.tight_layout()
plt.show()

## 8. Loss Computation

Compute each loss component and visualize what the model "sees".

In [ ]:
criterion = DeepVQELoss.from_config(cfg)
clean_wav = sample["clean_wav"].unsqueeze(0)

with torch.no_grad():
    total_loss, components = criterion(enhanced, clean_stft, clean_wav)

print("Loss components:")
for k, v in components.items():
    print(f"  {k:10s}: {v.item():.6f}")

# Visualize power-law compressed space
c = cfg.loss.power_law_c
pred_mag = torch.sqrt(enhanced[0, ..., 0]**2 + enhanced[0, ..., 1]**2 + 1e-12)
clean_mag_l = torch.sqrt(clean_stft[0, ..., 0]**2 + clean_stft[0, ..., 1]**2 + 1e-12)

pred_compressed = pred_mag.pow(c)
clean_compressed = clean_mag_l.pow(c)
error_map = (pred_compressed - clean_compressed).abs()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

im0 = axes[0].imshow(pred_compressed.numpy(), aspect="auto", origin="lower", cmap="magma")
axes[0].set_title(f"Enhanced |X|^{c}")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(clean_compressed.numpy(), aspect="auto", origin="lower", cmap="magma")
axes[1].set_title(f"Clean |X|^{c}")
fig.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(error_map.numpy(), aspect="auto", origin="lower", cmap="hot")
axes[2].set_title("Magnitude Error Map")
fig.colorbar(im2, ax=axes[2])

for ax in axes:
    ax.set_xlabel("Frame")
    ax.set_ylabel("Freq bin")

fig.suptitle("Power-Law Compressed Loss Space", y=1.01)
fig.tight_layout()
plt.show()

# Time-domain error
enh_wav = istft(enhanced, cfg.audio.n_fft, cfg.audio.hop_length,
                length=clean_wav.shape[-1])
error_wav = (enh_wav - clean_wav)[0].numpy()

fig, ax = plt.subplots(1, 1, figsize=(14, 3))
ax.plot(np.arange(len(error_wav)) / sr, error_wav, linewidth=0.5, color="crimson")
ax.set_title(f"Time-Domain Error (L1={np.abs(error_wav).mean():.6f})")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Error")
fig.tight_layout()
plt.show()

## 9. Spectrogram Comparison (mic vs enhanced vs clean)

In [ ]:
fig = plot_spectrogram_comparison(
    mic_stft, enhanced, clean_stft, sr, cfg.audio.hop_length
)
plt.show()

## 10. Audio Playback

Listen to mic, enhanced, and clean signals side by side.

In [ ]:
mic_wav_np = sample["mic_wav"].numpy()
clean_wav_np = sample["clean_wav"].numpy()
enh_wav_np = enh_wav[0].detach().numpy()

print("Microphone (noisy + echo):")
ipd.display(ipd.Audio(mic_wav_np, rate=sr))

print("\nEnhanced:")
ipd.display(ipd.Audio(enh_wav_np, rate=sr))

print("\nClean (target):")
ipd.display(ipd.Audio(clean_wav_np, rate=sr))

# Quick spectrograms next to audio
fig, axes = plt.subplots(1, 3, figsize=(16, 3))
for ax, wav, title in zip(axes, [mic_wav_np, enh_wav_np, clean_wav_np],
                           ["Mic", "Enhanced", "Clean"]):
    spec = np.abs(np.fft.rfft(wav.reshape(-1, cfg.audio.hop_length), n=cfg.audio.n_fft, axis=-1).T)
    ax.imshow(20*np.log10(spec + 1e-12), aspect="auto", origin="lower", cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("Frame")
    ax.set_ylabel("Freq bin")
fig.tight_layout()
plt.show()

In [ ]:
# Cleanup hooks
remove_hooks(hook_handles)
print("Done! Hooks removed.")